
# 05 — Rekomendasi LLM dan Evaluasi Final

Notebook ini menjalankan tahap **LLM-assisted recommendation** setelah hasil K-Means final selesai.

## Posisi LLM

LLM tidak digunakan untuk menentukan jumlah cluster, mengubah hasil K-Means, mengganti label sentimen, atau menjadi sumber utama kebenaran analitik. LLM hanya membantu memformulasikan rekomendasi berdasarkan hasil analisis yang sudah tersedia.

## Input

`04_Hasil_Clustering_Final.csv`

## Jumlah rekomendasi

Lima listing dipilih dari setiap cluster:

```text
3 cluster × 5 listing = 15 rekomendasi
```

Sampling memakai `random_state=42` dan memprioritaskan keberagaman kategori.

## Output

- `05_Rekomendasi_LLM_15_Sampel.csv`
- `05_Hasil_Rekomendasi_dan_Evaluasi_LLM.xlsx`
- `05_Checkpoint_Rekomendasi_LLM.csv`


## Konsistensi nama cluster

Notebook ini hanya menerima tiga nama cluster final:

1. `Reputasi Positif–Volume Ulasan Rendah`
2. `Reputasi Digital Perlu Perbaikan`
3. `Performa Digital Tinggi`

Dataset dengan nama cluster versi lama akan ditolak agar output modeling dan LLM tidak tercampur.


## Versi prompt final

Notebook menggunakan `PROMPT_VERSION = "V3_FACT_LOCKED"`. Versi ini mengunci fakta numerik melalui Python, mencatat versi prompt pada checkpoint, dan menolak penggunaan checkpoint dari versi prompt lain.


In [ ]:

# ============================================================
# 1. INSTALASI
# ============================================================
!pip install -q -U openai openpyxl


In [ ]:

# ============================================================
# 2. IMPORT
# ============================================================
import os
import re
import time
import pandas as pd

from openai import (
    OpenAI,
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
    APIStatusError,
)

try:
    from google.colab import files, userdata
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("Library berhasil dimuat.")


Library berhasil dimuat.


In [ ]:

# ============================================================
# 3. KONFIGURASI
# ============================================================
INPUT_FILE = "04_Hasil_Clustering_Final.csv"

OUTPUT_CSV = "05_Rekomendasi_LLM_15_Sampel_V3.csv"
OUTPUT_EVAL_CSV = "05_Hasil_Rekomendasi_dan_Evaluasi_LLM_V3.csv"
OUTPUT_CHECKPOINT = "05_Checkpoint_Rekomendasi_LLM_V3.csv"

MODEL_NAME = "qwen/qwen-2.5-7b-instruct"

# Penanda versi prompt untuk audit dan mencegah checkpoint lama tercampur.
PROMPT_VERSION = "V3_FACT_LOCKED"

N_SAMPLE_PER_CLUSTER = 5
RANDOM_STATE = 42

TEMPERATURE = 0.25
MAX_TOKENS = 700
MAX_RETRIES = 4
BASE_WAIT_SECONDS = 3

# Pilihan paling konservatif:
# metadata audit NLP tetap diekspor, tetapi tidak dijadikan isi prompt.
USE_NLP_AUDIT_IN_PROMPT = False

EXPECTED_ROWS = 3438
EXPECTED_CLUSTERS = 3
EXPECTED_RECOMMENDATIONS = 15

print("Model:", MODEL_NAME)
print("Prompt version:", PROMPT_VERSION)
print("Target rekomendasi:", EXPECTED_RECOMMENDATIONS)


Model: qwen/qwen-2.5-7b-instruct
Prompt version: V3_FACT_LOCKED
Target rekomendasi: 15


In [ ]:

# ============================================================
# 4. UPLOAD INPUT
# ============================================================
if not os.path.exists(INPUT_FILE):
    if not RUNNING_IN_COLAB:
        raise FileNotFoundError(INPUT_FILE)

    print("Upload file:", INPUT_FILE)
    uploaded = files.upload()

    if INPUT_FILE not in uploaded:
        csv_files = [
            name for name in uploaded
            if name.lower().endswith(".csv")
        ]

        if len(csv_files) != 1:
            raise FileNotFoundError(
                "Upload tepat satu file CSV hasil clustering."
            )

        os.rename(csv_files[0], INPUT_FILE)

print("File ditemukan:", INPUT_FILE)


File ditemukan: 04_Hasil_Clustering_Final.csv


In [ ]:

# ============================================================
# 5. MEMBACA DAN MEMVALIDASI DATA
# ============================================================
df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
    on_bad_lines="warn",
)

required_columns = [
    "entity_key", "title", "totalScore", "reviewsCount",
    "street", "city", "categoryName", "sentiment_score",
    "sentimen_confidence", "cluster_id", "cluster_name",
    "cluster_short_name", "nlp_review_flag", "nlp_review_reason",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom wajib tidak tersedia: {missing_columns}"
    )

df = df[required_columns].copy()

df["totalScore"] = pd.to_numeric(
    df["totalScore"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)

df["reviewsCount"] = pd.to_numeric(
    df["reviewsCount"],
    errors="coerce",
)

df["sentiment_score"] = pd.to_numeric(
    df["sentiment_score"],
    errors="coerce",
).astype("Int64")

df["cluster_id"] = pd.to_numeric(
    df["cluster_id"],
    errors="coerce",
).astype("Int64")

df["nlp_review_flag"] = (
    df["nlp_review_flag"]
    .fillna(False)
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    })
    .fillna(False)
)

assert len(df) == EXPECTED_ROWS
assert df["entity_key"].is_unique
assert df["cluster_id"].nunique() == EXPECTED_CLUSTERS
assert df["cluster_id"].isin([1, 2, 3]).all()
assert df["sentiment_score"].isin([-1, 0, 1]).all()
assert df["totalScore"].between(1, 5, inclusive="both").all()
assert (df["reviewsCount"] >= 1).all()

EXPECTED_CLUSTER_NAMES = {
    "Reputasi Positif–Volume Ulasan Rendah",
    "Reputasi Digital Perlu Perbaikan",
    "Performa Digital Tinggi",
}

EXPECTED_CLUSTER_SHORT_NAMES = {
    "Positive Reputation–Low Review Volume",
    "Digital Reputation Needs Improvement",
    "High Digital Performance",
}

actual_cluster_names = set(
    df["cluster_name"].dropna().unique()
)

actual_short_names = set(
    df["cluster_short_name"].dropna().unique()
)

if actual_cluster_names != EXPECTED_CLUSTER_NAMES:
    raise ValueError(
        "Nama cluster pada input belum memakai versi final. "
        "Jalankan ulang notebook modeling V4, lalu upload ulang "
        "04_Hasil_Clustering_Final.csv. "
        f"Nama yang ditemukan: {sorted(actual_cluster_names)}"
    )

if actual_short_names != EXPECTED_CLUSTER_SHORT_NAMES:
    raise ValueError(
        "cluster_short_name belum konsisten. "
        "Jalankan ulang notebook modeling V4."
    )

print("Jumlah data:", len(df))
display(
    df.groupby(["cluster_id", "cluster_name"])
    .size()
    .rename("jumlah")
    .to_frame()
)


Jumlah data: 3438


,,jumlah
cluster_id,cluster_name,
1,Reputasi Positif–Volume Ulasan Rendah,1748
2,Reputasi Digital Perlu Perbaikan,435
3,Performa Digital Tinggi,1255


In [ ]:

# ============================================================
# 6. PROFIL CLUSTER DARI DATA AKTUAL
# ============================================================
cluster_profile = (
    df.groupby(
        ["cluster_id", "cluster_name"],
        as_index=False,
    )
    .agg(
        jumlah_usaha=("entity_key", "size"),
        rating_mean=("totalScore", "mean"),
        rating_median=("totalScore", "median"),
        reviews_mean=("reviewsCount", "mean"),
        reviews_median=("reviewsCount", "median"),
        sentiment_mean=("sentiment_score", "mean"),
    )
    .sort_values("cluster_id")
)

cluster_profile["persentase"] = (
    cluster_profile["jumlah_usaha"]
    / len(df)
    * 100
)

cluster_profile_lookup = {
    row["cluster_name"]: row.to_dict()
    for _, row in cluster_profile.iterrows()
}

display(cluster_profile)


,cluster_id,cluster_name,jumlah_usaha,rating_mean,rating_median,reviews_mean,reviews_median,sentiment_mean,persentase
0,1,Reputasi Positif–Volume Ulasan Rendah,1748,4.700686,4.8,18.771167,13.0,0.960526,50.843514
1,2,Reputasi Digital Perlu Perbaikan,435,3.979770,4.2,142.043678,36.0,-0.970115,12.652705
2,3,Performa Digital Tinggi,1255,4.625657,4.6,830.650199,232.0,0.996813,36.503781


In [ ]:
SENTIMENT_LABEL = {
    -1: "negatif",
    0: "netral",
    1: "positif",
}


def compare_review_volume(
    review_count: int,
    cluster_median: float,
) -> str:
    """Membandingkan volume ulasan tanpa menyerahkan aritmetika ke LLM."""
    if review_count < cluster_median:
        return "lebih rendah daripada median cluster"

    if review_count > cluster_median:
        return "lebih tinggi daripada median cluster"

    return "sama dengan median cluster"


def build_locked_facts(row: pd.Series) -> dict:
    cluster_name = row["cluster_name"]
    profile = cluster_profile_lookup[cluster_name]

    review_count = int(row["reviewsCount"])
    cluster_median = float(
        profile["reviews_median"]
    )

    sentiment_score = int(
        row["sentiment_score"]
    )

    return {
        "title": row["title"],
        "category": row["categoryName"],
        "rating": float(row["totalScore"]),
        "review_count": review_count,
        "cluster_median_reviews": cluster_median,
        "review_position": compare_review_volume(
            review_count,
            cluster_median,
        ),
        "sentiment_score": sentiment_score,
        "sentiment_label": SENTIMENT_LABEL[
            sentiment_score
        ],
        "review_coverage": row[
            "sentimen_confidence"
        ],
        "cluster_name": cluster_name,
    }

In [ ]:

# ============================================================
# 7. CLUSTER CONTEXT DAN THEORY BANK
# ============================================================
CLUSTER_CONTEXT = {
    "Reputasi Positif–Volume Ulasan Rendah": {
        "karakteristik": (
            "Rating dan hasil sentiment analysis cenderung positif, "
            "tetapi volume ulasan Google Maps relatif lebih rendah "
            "dibandingkan cluster lain dalam dataset. Kondisi ini hanya "
            "menunjukkan jejak ulasan digital yang lebih terbatas dan "
            "tidak membuktikan jumlah pelanggan, tingkat keramaian, "
            "popularitas offline, kunjungan, transaksi, atau penjualan rendah."
        ),
        "prioritas": (
            "menjaga kualitas yang telah memperoleh penilaian positif, "
            "memastikan informasi profil Google Maps akurat dan mudah dipahami, "
            "serta memudahkan pelanggan asli yang bersedia memberikan "
            "ulasan jujur tanpa paksaan atau imbalan. Penguatan jejak ulasan "
            "bersifat pilihan, bukan karena usaha diasumsikan sepi."
        ),
        "teori_relevan": [
            "Electronic Word-of-Mouth (eWOM)",
            "Digital Marketing Adoption",
            "Customer Engagement",
        ],
    },
    "Reputasi Digital Perlu Perbaikan": {
        "karakteristik": (
            "Hasil sentimen teks relatif lebih rendah atau negatif. "
            "Cluster ini tidak boleh langsung disebut sebagai kelompok usaha "
            "yang buruk karena rating, jumlah teks, bahasa daerah, slang, "
            "dan keterbatasan NLP tetap harus dipertimbangkan."
        ),
        "prioritas": (
            "memeriksa tema keluhan secara manual sebelum mengambil tindakan, "
            "memperbaiki aspek layanan yang benar-benar terkonfirmasi, "
            "merespons keluhan secara etis, dan melakukan pemulihan layanan"
        ),
        "teori_relevan": [
            "SERVQUAL",
            "Service Recovery Paradox",
            "Customer Satisfaction Theory",
        ],
    },
    "Performa Digital Tinggi": {
        "karakteristik": (
            "Rating baik, hasil sentimen cenderung positif, dan volume ulasan "
            "Google Maps relatif tinggi. Istilah ini hanya menggambarkan "
            "performa digital pada dataset, bukan omzet, laba, jumlah pelanggan, "
            "pangsa pasar, atau kepemimpinan pasar."
        ),
        "prioritas": (
            "menjaga konsistensi kualitas, mempertahankan pengalaman pelanggan, "
            "mengelola kapasitas layanan sesuai kondisi nyata usaha, dan "
            "memanfaatkan reputasi digital secara berkelanjutan"
        ),
        "teori_relevan": [
            "Customer Loyalty",
            "Customer Experience",
            "Electronic Word-of-Mouth (eWOM)",
        ],
    },
}

THEORY_BANK = {
    "Electronic Word-of-Mouth (eWOM)": (
        "Ulasan dan rekomendasi pelanggan pada platform digital dapat "
        "memengaruhi persepsi serta keputusan calon pelanggan."
    ),
    "Digital Marketing Adoption": (
        "Pemanfaatan kanal digital perlu disesuaikan dengan kemampuan, "
        "kebutuhan, dan karakteristik usaha."
    ),
    "Customer Engagement": (
        "Interaksi yang relevan dan berkelanjutan dapat memperkuat "
        "hubungan pelanggan dengan usaha."
    ),
    "SERVQUAL": (
        "Kualitas layanan dapat ditinjau melalui reliability, responsiveness, "
        "assurance, empathy, dan tangibles."
    ),
    "Service Recovery Paradox": (
        "Penanganan kegagalan layanan secara cepat, adil, dan bertanggung jawab "
        "dapat membantu memulihkan kepuasan serta kepercayaan."
    ),
    "Customer Satisfaction Theory": (
        "Kepuasan terbentuk ketika pengalaman aktual memenuhi atau "
        "melampaui harapan pelanggan."
    ),
    "Customer Loyalty": (
        "Loyalitas dipengaruhi oleh pengalaman positif, kepuasan, "
        "kepercayaan, dan konsistensi hubungan."
    ),
    "Customer Experience": (
        "Pengalaman pelanggan mencakup seluruh interaksi sebelum, saat, "
        "dan setelah transaksi."
    ),
    "Trust": (
        "Kepercayaan tumbuh melalui konsistensi, transparansi, kompetensi, "
        "dan pemenuhan janji layanan."
    ),
    "Relationship Marketing": (
        "Hubungan jangka panjang dibangun melalui komunikasi, pelayanan "
        "konsisten, dan perhatian terhadap kebutuhan pelanggan."
    ),
}

assert set(CLUSTER_CONTEXT) == set(df["cluster_name"].unique())

print("Cluster context dan theory bank siap.")


Cluster context dan theory bank siap.


In [ ]:

# ============================================================
# 8. KONTEKS KATEGORI USAHA
# ============================================================
def get_category_context(category):
    category_lower = str(category).strip().lower()

    if any(k in category_lower for k in [
        "fotokopi", "foto kopi", "percetakan", "printing",
    ]):
        return {
            "jenis": "Jasa Percetakan/Fotokopi",
            "fokus": (
                "kecepatan pengerjaan, akurasi hasil, kualitas cetak, "
                "kejelasan harga, antrean, dan kelengkapan layanan"
            ),
            "teori_relevan": [
                "SERVQUAL",
                "Customer Satisfaction Theory",
            ],
        }

    if any(k in category_lower for k in [
        "bengkel", "motor", "mobil", "ban", "otomotif",
        "servis kendaraan", "cuci mobil", "cuci motor",
    ]):
        return {
            "jenis": "Otomotif/Bengkel",
            "fokus": (
                "transparansi diagnosis dan biaya, ketepatan pengerjaan, "
                "kompetensi teknis, garansi, keamanan, dan kepercayaan"
            ),
            "teori_relevan": [
                "Trust", "SERVQUAL", "Service Recovery Paradox",
            ],
        }

    if any(k in category_lower for k in [
        "salon", "barber", "cukur", "kecantikan",
        "spa", "nail", "perawatan rambut",
    ]):
        return {
            "jenis": "Kecantikan/Perawatan Pribadi",
            "fokus": (
                "konsistensi hasil, kebersihan alat, konsultasi kebutuhan, "
                "kenyamanan, keramahan, dan ketepatan jadwal"
            ),
            "teori_relevan": [
                "Customer Experience", "SERVQUAL", "Customer Loyalty",
            ],
        }

    if any(k in category_lower for k in [
        "laundry", "binatu", "cuci pakaian",
        "dry clean", "pembersih karpet",
    ]):
        return {
            "jenis": "Laundry/Kebersihan",
            "fokus": (
                "ketepatan waktu, kebersihan dan keamanan barang, "
                "transparansi harga, antar-jemput, dan penanganan kerusakan"
            ),
            "teori_relevan": [
                "SERVQUAL", "Service Recovery Paradox", "Trust",
            ],
        }

    if any(k in category_lower for k in [
        "apotek", "farmasi", "obat", "kesehatan",
        "laboratorium", "alat kesehatan",
    ]):
        return {
            "jenis": "Kesehatan/Apotek",
            "fokus": (
                "kepercayaan, ketersediaan produk, kecepatan pelayanan, "
                "kebersihan, keramahan, dan akurasi informasi nonmedis"
            ),
            "teori_relevan": [
                "Trust", "SERVQUAL", "Customer Experience",
            ],
        }

    if any(k in category_lower for k in [
        "komputer", "laptop", "elektronik", "ponsel",
        "handphone", "aksesori elektronik", "service komputer",
    ]):
        return {
            "jenis": "Elektronik/Komputer",
            "fokus": (
                "keaslian produk, transparansi spesifikasi, garansi, "
                "dukungan purnajual, keamanan transaksi, dan stok"
            ),
            "teori_relevan": [
                "Trust", "Customer Satisfaction Theory", "Customer Loyalty",
            ],
        }

    if any(k in category_lower for k in [
        "bahan kue", "bahan makanan", "peralatan",
        "perlengkapan restoran", "grosir", "kelontong",
        "sembako", "makanan beku",
    ]):
        return {
            "jenis": "Retail Bahan/Perlengkapan",
            "fokus": (
                "kelengkapan dan kesegaran stok, harga, kemudahan pembelian, "
                "akses, dan hubungan dengan pelanggan reguler"
            ),
            "teori_relevan": [
                "Customer Loyalty", "Relationship Marketing",
                "Customer Engagement",
            ],
        }

    if any(k in category_lower for k in [
        "restoran", "rumah makan", "warung", "kedai",
        "cafe", "kafe", "kopi", "bakso", "nasi",
        "ayam", "mie", "soto", "roti", "kue",
        "bakery", "minuman", "kuliner", "sate",
    ]):
        return {
            "jenis": "Kuliner",
            "fokus": (
                "rasa, kebersihan, konsistensi menu, kecepatan pelayanan, "
                "penyajian, kenyamanan, dan pengalaman makan"
            ),
            "teori_relevan": [
                "Electronic Word-of-Mouth (eWOM)",
                "SERVQUAL",
                "Customer Satisfaction Theory",
            ],
        }

    if any(k in category_lower for k in [
        "pakaian", "fashion", "sepatu", "tas", "batik",
        "aksesoris", "tekstil", "seragam", "distro",
    ]):
        return {
            "jenis": "Fashion",
            "fokus": (
                "kualitas dan variasi produk, ukuran dan stok, display, "
                "pelayanan konsultatif, penukaran, dan pembelian ulang"
            ),
            "teori_relevan": [
                "Customer Experience", "Customer Loyalty",
                "Electronic Word-of-Mouth (eWOM)",
            ],
        }

    if any(k in category_lower for k in [
        "toko", "retail", "ritel", "outlet",
        "market", "alat tulis", "mebel", "furnitur",
    ]):
        return {
            "jenis": "Retail/Toko",
            "fokus": (
                "kelengkapan produk, harga, display, stok, "
                "kemudahan akses, pelayanan, dan pembelian ulang"
            ),
            "teori_relevan": [
                "Customer Loyalty", "Customer Engagement",
                "Digital Marketing Adoption",
            ],
        }

    if any(k in category_lower for k in [
        "jasa", "reparasi", "jahit", "rental",
        "ekspedisi", "kurir", "studio",
    ]):
        return {
            "jenis": "Jasa",
            "fokus": (
                "ketepatan waktu, kejelasan harga, kualitas hasil, "
                "respons komunikasi, penanganan komplain, dan kepercayaan"
            ),
            "teori_relevan": [
                "SERVQUAL", "Trust", "Customer Satisfaction Theory",
            ],
        }

    return {
        "jenis": "Usaha Lokal Umum",
        "fokus": (
            "kualitas produk atau layanan, transparansi, pengalaman pelanggan, "
            "reputasi digital, dan hubungan pelanggan"
        ),
        "teori_relevan": [
            "Customer Satisfaction Theory",
            "Customer Experience",
            "Electronic Word-of-Mouth (eWOM)",
        ],
    }


category_tests = {
    "Toko Peralatan dan Bahan Kue": "Retail Bahan/Perlengkapan",
    "Toko Fotokopi": "Jasa Percetakan/Fotokopi",
    "Toko Kue": "Kuliner",
    "Bengkel Sepeda Motor": "Otomotif/Bengkel",
    "Tempat Cukur Rambut": "Kecantikan/Perawatan Pribadi",
    "Layanan Binatu": "Laundry/Kebersihan",
}

for category, expected_type in category_tests.items():
    actual_type = get_category_context(category)["jenis"]
    assert actual_type == expected_type

print("Konteks kategori berhasil dibuat dan diuji.")


Konteks kategori berhasil dibuat dan diuji.


In [ ]:

# ============================================================
# 9. MEMILIH 5 SAMPEL PER CLUSTER
# ============================================================
def diverse_sample(group, n=5, seed=42):
    shuffled = group.sample(frac=1, random_state=seed)

    selected_indices = []
    used_categories = set()

    for index, row in shuffled.iterrows():
        category = str(row["categoryName"]).strip().lower()

        if category not in used_categories:
            selected_indices.append(index)
            used_categories.add(category)

        if len(selected_indices) == n:
            break

    if len(selected_indices) < n:
        remaining = shuffled.loc[
            ~shuffled.index.isin(selected_indices)
        ]

        selected_indices.extend(
            remaining.head(
                n - len(selected_indices)
            ).index.tolist()
        )

    return group.loc[selected_indices]


sample_parts = []

for _, group in df.groupby("cluster_id"):
    sample_parts.append(
        diverse_sample(
            group,
            n=N_SAMPLE_PER_CLUSTER,
            seed=RANDOM_STATE,
        )
    )

sample_eval = pd.concat(
    sample_parts,
    ignore_index=True,
)

sample_eval["category_context"] = sample_eval[
    "categoryName"
].apply(
    lambda value: get_category_context(value)["jenis"]
)

assert len(sample_eval) == EXPECTED_RECOMMENDATIONS
assert sample_eval["entity_key"].is_unique
assert sample_eval["cluster_id"].value_counts().eq(5).all()

display(
    sample_eval[
        [
            "title", "categoryName", "category_context",
            "totalScore", "reviewsCount", "sentiment_score",
            "sentimen_confidence", "cluster_id", "cluster_name",
            "nlp_review_flag",
        ]
    ]
)


,title,categoryName,category_context,totalScore,reviewsCount,sentiment_score,sentimen_confidence,cluster_id,cluster_name,nlp_review_flag
0,XT komputer,Toko Komputer,Elektronik/Komputer,3.8,32,1,Tinggi,1,Reputasi Positif–Volume Ulasan Rendah,False
1,BMW Motorrad DC Motorindo Bandung,Dealer Sepeda Motor BMW,Otomotif/Bengkel,5.0,35,1,Tinggi,1,Reputasi Positif–Volume Ulasan Rendah,False
2,Toserba Dm.Giftidea,Toko Alat Tulis Kantor,Retail/Toko,4.3,17,1,Tinggi,1,Reputasi Positif–Volume Ulasan Rendah,False
3,De & Dy,Toko Sepatu,Fashion,4.9,10,1,Tinggi,1,Reputasi Positif–Volume Ulasan Rendah,False
4,Mutiara Salon,Salon Rambut,Kecantikan/Perawatan Pribadi,4.9,31,1,Tinggi,1,Reputasi Positif–Volume Ulasan Rendah,False
5,Aldy Motor,Bengkel Sepeda Motor,Otomotif/Bengkel,3.3,18,-1,Tinggi,2,Reputasi Digital Perlu Perbaikan,False
6,Laundry Club Antapani Kidul (Laundry Express Antar dan Jemput),Layanan Binatu,Laundry/Kebersihan,4.9,245,-1,Tinggi,2,Reputasi Digital Perlu Perbaikan,True
7,SADAYA EXPRESS HEAD OFFICE BANDUNG,Jasa Pengiriman,Jasa,3.8,6,-1,Rendah,2,Reputasi Digital Perlu Perbaikan,True
8,Seblak dondon,Restoran cepat saji,Kuliner,4.3,36,-1,Tinggi,2,Reputasi Digital Perlu Perbaikan,False
9,Bengkel Motor & Aksesoris — Berkah Pit Station,Toko Suku Cadang Motor,Otomotif/Bengkel,4.0,5,-1,Tinggi,2,Reputasi Digital Perlu Perbaikan,False


In [ ]:

# ============================================================
# 10. INSTRUKSI KEHATI-HATIAN DATA
# ============================================================
def get_data_caution(
    sentiment_confidence,
    nlp_review_flag=False,
    nlp_review_reason="",
):
    confidence = str(sentiment_confidence).strip().lower()

    if confidence == "rendah":
        instruction = (
            "Skor sentimen hanya berasal dari satu teks. "
            "Jangan menjadikannya dasar utama. Gunakan bahasa tidak mutlak "
            "dan sarankan pengumpulan ulasan tambahan."
        )
    elif confidence == "sedang":
        instruction = (
            "Skor sentimen berasal dari dua teks. Gunakan secara hati-hati, "
            "hindari kesimpulan mutlak, dan sarankan validasi tambahan."
        )
    else:
        instruction = (
            "Skor sentimen berasal dari minimal tiga teks dan dapat digunakan "
            "sebagai indikator tambahan, tetapi bukan kebenaran mutlak."
        )

    if USE_NLP_AUDIT_IN_PROMPT and bool(nlp_review_flag):
        reason = (
            str(nlp_review_reason).strip()
            if pd.notna(nlp_review_reason)
            else ""
        )

        instruction += (
            " Data memiliki indikator pemeriksaan NLP internal"
            + (f": {reason}." if reason else ".")
            + " Indikator ini bukan bukti bahwa usaha bermasalah. "
              "Jangan membuat klaim negatif berdasarkan indikator tersebut."
        )

    return instruction


for level in ["Rendah", "Sedang", "Tinggi"]:
    print(level, "->", get_data_caution(level))


Rendah -> Skor sentimen hanya berasal dari satu teks. Jangan menjadikannya dasar utama. Gunakan bahasa tidak mutlak dan sarankan pengumpulan ulasan tambahan.
Sedang -> Skor sentimen berasal dari dua teks. Gunakan secara hati-hati, hindari kesimpulan mutlak, dan sarankan validasi tambahan.
Tinggi -> Skor sentimen berasal dari minimal tiga teks dan dapat digunakan sebagai indikator tambahan, tetapi bukan kebenaran mutlak.


In [ ]:
# ============================================================
# 11. KONFIGURASI OPENROUTER MELALUI COLAB SECRETS
# ============================================================
if not RUNNING_IN_COLAB:
    raise RuntimeError(
        "Cell API dirancang untuk Google Colab."
    )

try:
    OPENROUTER_API_KEY = userdata.get(
        "OPENROUTER_API_KEY"
    )
except Exception as error:
    raise ValueError(
        "Secret OPENROUTER_API_KEY tidak ditemukan. "
        "Buka ikon kunci di sisi kiri Colab, buat secret "
        "bernama OPENROUTER_API_KEY, lalu izinkan akses notebook."
    ) from error

if not OPENROUTER_API_KEY:
    raise ValueError(
        "Secret OPENROUTER_API_KEY kosong."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    timeout=90.0,
    max_retries=0,
    default_headers={
        "HTTP-Referer": (
            "https://colab.research.google.com/"
        ),
        "X-Title": (
            "UMKM Bandung LLM Recommendation"
        ),
    },
)

print("Client OpenRouter siap.")
print("Model:", MODEL_NAME)

Client OpenRouter siap.
Model: qwen/qwen-2.5-7b-instruct


In [ ]:

# ============================================================
# 12. MEMBENTUK PROMPT REKOMENDASI
# ============================================================
def build_prompt(row: pd.Series) -> str:
    facts = build_locked_facts(row)

    cluster_info = CLUSTER_CONTEXT[
        facts["cluster_name"]
    ]

    category_info = get_category_context(
        facts["category"]
    )

    theories = list(
        dict.fromkeys(
            cluster_info["teori_relevan"]
            + category_info["teori_relevan"]
        )
    )[:4]

    theory_description = "\n".join(
        f"- {theory}: {THEORY_BANK[theory]}"
        for theory in theories
    )

    data_caution = get_data_caution(
        facts["review_coverage"]
    )

    return f"""
Anda membantu memformulasikan rekomendasi berdasarkan
hasil analisis yang telah dikunci oleh Python.

FAKTA LISTING — JANGAN DIUBAH:
- Nama: {facts['title']}
- Kategori: {facts['category']}
- Rating Google Maps: {facts['rating']} dari 5
- Volume ulasan Google Maps: {facts['review_count']}
- Median volume ulasan cluster:
  {facts['cluster_median_reviews']:.0f}
- Posisi volume ulasan yang sudah dihitung Python:
  {facts['review_position']}
- Sentiment score: {facts['sentiment_score']}
- Label sentimen yang sudah dipetakan Python:
  {facts['sentiment_label']}
- Tingkat kecukupan teks:
  {facts['review_coverage']}
- Cluster final:
  {facts['cluster_name']}

KARAKTERISTIK CLUSTER:
{cluster_info['karakteristik']}

ARAH UMUM CLUSTER:
{cluster_info['prioritas']}

KONTEKS OPERASIONAL KATEGORI:
{category_info['fokus']}

KEHATI-HATIAN DATA:
{data_caution}

TEORI YANG BOLEH DIGUNAKAN:
{theory_description}

ATURAN WAJIB:
1. Jangan menghitung ulang atau membantah fakta yang sudah
   dihitung Python.
2. Jangan menyebut rata-rata atau median kategori karena
   statistik kategori tidak diberikan.
3. Sentiment score 1 berarti positif, 0 netral,
   dan -1 negatif.
4. Tingkat kecukupan teks bukan akurasi atau confidence model.
5. Volume ulasan rendah bukan bukti usaha sepi,
   tidak populer, sedikit pelanggan, atau penjualan rendah.
6. Volume ulasan tinggi bukan bukti omzet tinggi,
   banyak pelanggan, atau dominasi pasar.
7. Jangan mengarang omzet, laba, jumlah pelanggan,
   kondisi internal, pangsa pasar, atau masalah operasional.
8. Gunakan istilah “hasil sentimen cenderung positif/netral/
   negatif”, bukan klaim bahwa seluruh pelanggan pasti puas.
9. Berikan tepat dua tindakan realistis untuk 30 hari.
10. Setiap tindakan wajib memiliki indikator yang dapat
    dihitung oleh usaha.
11. Jangan menyarankan pembelian atau manipulasi ulasan.
12. Gunakan maksimal dua teori.
13. Maksimal 350 kata.

STRUKTUR WAJIB:

[Analisis Konteks]

[Rekomendasi Strategi]
1. Tindakan dan indikator
2. Tindakan dan indikator

[Landasan Teori]

[Catatan Validitas Data]

Langsung jawab menggunakan empat bagian tersebut.
""".strip()


In [ ]:
# ============================================================
# 12B. VALIDASI OTOMATIS RESPONS LLM
# ============================================================
REQUIRED_SECTIONS = [
    "[Analisis Konteks]",
    "[Rekomendasi Strategi]",
    "[Landasan Teori]",
    "[Catatan Validitas Data]",
]

# Hanya klaim yang benar-benar berisiko.
# Jangan memasukkan frasa seperti "usaha sepi",
# karena dapat muncul dalam kalimat penyangkalan yang benar.
FORBIDDEN_PATTERNS = [
    r"\brata-rata kategori\b",
    r"\bmedian kategori\b",
    r"\bmarket leader\b",
    r"\bmendominasi pasar\b",
    r"\bseluruh pelanggan pasti puas\b",
    r"\bseluruh pelanggan pasti tidak puas\b",
    r"\bomzet usaha (tinggi|rendah)\b",
    r"\bjumlah pelanggan (tinggi|rendah)\b",
    r"\bsentimen (sangat )?akurat\b",
]


def normalize_for_validation(text):
    return (
        str(text)
        .lower()
        .replace("–", "-")
        .replace("—", "-")
    )


def validate_llm_response(
    text: str,
    row: pd.Series,
) -> list[str]:
    errors = []

    text_original = str(text)
    text_lower = normalize_for_validation(
        text_original
    )

    facts = build_locked_facts(row)

    # --------------------------------------------
    # 1. STRUKTUR WAJIB
    # --------------------------------------------
    for section in REQUIRED_SECTIONS:
        if section.lower() not in text_lower:
            errors.append(
                f"Bagian tidak tersedia: {section}"
            )

    # --------------------------------------------
    # 2. KLAIM TERLARANG
    # --------------------------------------------
    for pattern in FORBIDDEN_PATTERNS:
        if re.search(
            pattern,
            text_lower,
            flags=re.IGNORECASE,
        ):
            errors.append(
                f"Klaim terlarang ditemukan: {pattern}"
            )

    # --------------------------------------------
    # 3. PERBANDINGAN VOLUME ULASAN
    # --------------------------------------------
    review_position = facts[
        "review_position"
    ]

    if (
        review_position
        == "lebih tinggi daripada median cluster"
    ):
        wrong_comparison_patterns = [
            r"volume ulasan.{0,80}lebih rendah.{0,40}median",
            r"jumlah ulasan.{0,80}lebih rendah.{0,40}median",
            r"di bawah median cluster",
        ]

        for pattern in wrong_comparison_patterns:
            if re.search(
                pattern,
                text_lower,
                flags=re.DOTALL,
            ):
                errors.append(
                    "Perbandingan volume ulasan terbalik: "
                    "data berada di atas median cluster."
                )
                break

    elif (
        review_position
        == "lebih rendah daripada median cluster"
    ):
        wrong_comparison_patterns = [
            r"volume ulasan.{0,80}lebih tinggi.{0,40}median",
            r"jumlah ulasan.{0,80}lebih tinggi.{0,40}median",
            r"di atas median cluster",
        ]

        for pattern in wrong_comparison_patterns:
            if re.search(
                pattern,
                text_lower,
                flags=re.DOTALL,
            ):
                errors.append(
                    "Perbandingan volume ulasan terbalik: "
                    "data berada di bawah median cluster."
                )
                break

    # --------------------------------------------
    # 4. PEMBACAAN SENTIMENT SCORE
    # --------------------------------------------
    expected_label = facts[
        "sentiment_label"
    ]

    # Hanya mendeteksi klaim afirmatif yang jelas.
    # Kalimat "bukan sentimen negatif" tidak ditolak.
    affirmative_sentiment_patterns = {
        "positif": [
            r"(?:hasil|skor|label) sentimen(?:nya)? "
            r"(?:adalah|tergolong|menunjukkan) negatif",
        ],
        "netral": [
            r"(?:hasil|skor|label) sentimen(?:nya)? "
            r"(?:adalah|tergolong|menunjukkan) positif",
            r"(?:hasil|skor|label) sentimen(?:nya)? "
            r"(?:adalah|tergolong|menunjukkan) negatif",
        ],
        "negatif": [
            r"(?:hasil|skor|label) sentimen(?:nya)? "
            r"(?:adalah|tergolong|menunjukkan) positif",
        ],
    }

    for pattern in affirmative_sentiment_patterns[
        expected_label
    ]:
        if re.search(
            pattern,
            text_lower,
            flags=re.IGNORECASE,
        ):
            errors.append(
                "LLM salah membaca label sentimen."
            )
            break

    # --------------------------------------------
    # 5. KECUKUPAN TEKS BUKAN AKURASI
    # --------------------------------------------
    confidence_error_patterns = [
        r"kecukupan teks.{0,50}"
        r"(menjamin|membuktikan).{0,30}"
        r"(akurasi|kebenaran)",
        r"tingkat kecukupan.{0,40}"
        r"berarti model akurat",
        r"confidence tinggi.{0,40}"
        r"berarti sentimen akurat",
    ]

    for pattern in confidence_error_patterns:
        if re.search(
            pattern,
            text_lower,
            flags=re.DOTALL,
        ):
            errors.append(
                "Tingkat kecukupan teks keliru "
                "dianggap sebagai akurasi model."
            )
            break

    return list(dict.fromkeys(errors))

In [ ]:
# ============================================================
# 12C. GENERATE DENGAN VALIDASI DAN RETRY
# ============================================================
def generate_validated_recommendation(
    row: pd.Series,
) -> dict:
    base_prompt = build_prompt(row)
    prompt = base_prompt
    last_validation_errors = []

    for validation_attempt in range(1, 4):
        result = call_llm(prompt)

        if result["status_api"] != "SUCCESS":
            continue

        validation_errors = validate_llm_response(
            result["rekomendasi_llm"],
            row,
        )

        if not validation_errors:
            result["validation_status"] = "VALID"
            result["validation_errors"] = ""
            result["validation_attempt"] = validation_attempt
            return result

        last_validation_errors = validation_errors

        prompt = (
            base_prompt
            + "\n\nRESPONS SEBELUMNYA TIDAK VALID.\n"
            + "Kesalahan yang wajib diperbaiki:\n- "
            + "\n- ".join(validation_errors)
            + "\nBuat ulang seluruh jawaban dari awal dan patuhi semua fakta."
        )

    return {
        "status_api": "FAILED_VALIDATION",
        "rekomendasi_llm": "",
        "error_api": (
            "Respons gagal memenuhi validasi otomatis "
            "setelah tiga percobaan validasi."
        ),
        "attempt_count": MAX_RETRIES,
        "response_model": MODEL_NAME,
        "generation_id": "",
        "validation_status": "INVALID",
        "validation_errors": " | ".join(
            last_validation_errors
        ),
        "validation_attempt": 3,
    }

In [ ]:
# ============================================================
# 13. FUNGSI API DENGAN RETRY YANG LEBIH AMAN
# ============================================================
def call_llm(prompt):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(
                f"Memanggil API, percobaan "
                f"{attempt}/{MAX_RETRIES}..."
            )

            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Anda adalah asisten formulasi rekomendasi "
                            "berbasis hasil clustering. Jangan mengubah "
                            "cluster, jangan mengarang data, dan selalu "
                            "nyatakan keterbatasan data."
                        ),
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                timeout=90,
            )

            # ------------------------------------------------
            # VALIDASI STRUKTUR RESPONS
            # ------------------------------------------------
            if response is None:
                raise ValueError(
                    "Objek response dari API bernilai None."
                )

            choices = getattr(
                response,
                "choices",
                None,
            )

            if not choices:
                raise ValueError(
                    "API tidak mengembalikan choices."
                )

            first_choice = choices[0]

            if first_choice is None:
                raise ValueError(
                    "choices[0] bernilai None."
                )

            message = getattr(
                first_choice,
                "message",
                None,
            )

            if message is None:
                raise ValueError(
                    "API tidak mengembalikan message."
                )

            content = getattr(
                message,
                "content",
                None,
            )

            if content is None:
                # Beberapa respons mungkin mempunyai refusal,
                # reasoning, atau struktur lain tanpa content.
                refusal = getattr(
                    message,
                    "refusal",
                    None,
                )

                reasoning = getattr(
                    message,
                    "reasoning",
                    None,
                )

                raise ValueError(
                    "Message tidak memiliki content. "
                    f"refusal={refusal}, "
                    f"reasoning tersedia={bool(reasoning)}"
                )

            if not isinstance(content, str):
                raise ValueError(
                    "Content respons bukan teks string. "
                    f"Tipe content: {type(content).__name__}"
                )

            content = content.strip()

            if not content:
                raise ValueError(
                    "Respons model kosong."
                )

            finish_reason = getattr(
                first_choice,
                "finish_reason",
                "",
            )

            print(
                "Respons berhasil diterima. "
                f"Finish reason: {finish_reason}"
            )

            return {
                "status_api": "SUCCESS",
                "rekomendasi_llm": content,
                "error_api": "",
                "attempt_count": attempt,
                "response_model": getattr(
                    response,
                    "model",
                    MODEL_NAME,
                ),
                "generation_id": getattr(
                    response,
                    "id",
                    "",
                ),
                "finish_reason": (
                    finish_reason
                    if finish_reason is not None
                    else ""
                ),
            }

        except (
            RateLimitError,
            APIConnectionError,
            APITimeoutError,
            APIStatusError,
            ValueError,
            TypeError,
            IndexError,
            AttributeError,
        ) as error:
            last_error = error

            print(
                f"Percobaan {attempt} gagal: "
                f"{type(error).__name__}: {error}"
            )

            if attempt < MAX_RETRIES:
                wait_seconds = (
                    BASE_WAIT_SECONDS
                    * (2 ** (attempt - 1))
                )

                print(
                    f"Menunggu {wait_seconds} detik "
                    "sebelum mencoba kembali..."
                )

                time.sleep(wait_seconds)

    return {
        "status_api": "FAILED",
        "rekomendasi_llm": "",
        "error_api": (
            f"{type(last_error).__name__}: "
            f"{last_error}"
        ),
        "attempt_count": MAX_RETRIES,
        "response_model": MODEL_NAME,
        "generation_id": "",
        "finish_reason": "",
    }


print("Fungsi API siap.")

Fungsi API siap.


In [ ]:

# ============================================================
# 14. UJI SATU REQUEST
# ============================================================
test_row = sample_eval.iloc[0]
test_prompt = build_prompt(test_row)

print("Sampel uji:", test_row["title"])
print("Cluster:", test_row["cluster_name"])
print("Kategori:", test_row["categoryName"])

test_result = generate_validated_recommendation(test_row)

print("Status API:", test_result["status_api"])
print(
    "Status validasi:",
    test_result.get("validation_status", "")
)
print(
    "Percobaan validasi:",
    test_result.get("validation_attempt", "")
)
print(
    "Kesalahan validasi:",
    test_result.get("validation_errors", "")
)

if test_result["status_api"] == "SUCCESS":
    print("\nHASIL UJI:\n")
    print(test_result["rekomendasi_llm"])
else:
    print("\nHASIL BELUM LOLOS VALIDASI:\n")

    print(
        "Error API:",
        test_result.get("error_api", "")
    )

    print(
        "Kesalahan validasi:",
        test_result.get(
            "validation_errors",
            ""
        )
    )

    raise RuntimeError(
        "API berhasil merespons, tetapi rekomendasi "
        "belum lolos validasi otomatis. "
        "Periksa validation_errors sebelum batch."
    )


Sampel uji: XT komputer
Cluster: Reputasi Positif–Volume Ulasan Rendah
Kategori: Toko Komputer
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Status API: SUCCESS
Status validasi: VALID
Percobaan validasi: 3
Kesalahan validasi: 

HASIL UJI:

[Analisis Konteks]

XT komputer memiliki reputasi yang positif, dengan rating 3.8 dari 5 dan sentimen ulasan yang cenderung positif. Meski demikian, volume ulasan yang lebih tinggi dari median cluster menunjukkan bahwa jejak ulasan digital XT komputer masih terbatas. Hal ini penting untuk diperhatikan karena volume ulasan yang lebih tinggi biasanya menunjukkan tingkat kepuasan pelanggan yang lebih tinggi dan kepercayaan yang lebih besar.

[Rekomendasi Strategi]
1. **Meningkatkan Volume Ulasan**
   - **Tindakan:** Mendorong pelanggan yang puas untuk memberikan ula


## Pemeriksaan sebelum batch

Pastikan hasil uji:

- mempunyai empat bagian;
- sesuai kategori;
- tidak mengarang omzet atau pangsa pasar;
- tidak mengubah cluster;
- membahas tingkat kecukupan teks;
- tidak menyarankan manipulasi ulasan.


In [ ]:
# ============================================================
# 15. MEMUAT CHECKPOINT
# ============================================================
checkpoint_columns = [
    "prompt_version",
    "entity_key",
    "cluster_id",
    "cluster_name",
    "title",
    "categoryName",
    "totalScore",
    "reviewsCount",
    "sentiment_score",
    "sentimen_confidence",
    "nlp_review_flag",
    "nlp_review_reason",
    "category_context",
    "prompt_llm",
    "status_api",
    "validation_status",
    "validation_errors",
    "validation_attempt",
    "rekomendasi_llm",
    "error_api",
    "attempt_count",
    "response_model",
    "generation_id",
]

if os.path.exists(OUTPUT_CHECKPOINT):
    checkpoint = pd.read_csv(
        OUTPUT_CHECKPOINT,
        sep=";",
        encoding="utf-8-sig",
        decimal=",",
    )

    # Checkpoint versi berbeda tidak boleh digunakan.
    if "prompt_version" not in checkpoint.columns:
        checkpoint = pd.DataFrame(
            columns=checkpoint_columns
        )
        print(
            "Checkpoint lama diabaikan karena tidak memiliki prompt_version."
        )
    else:
        checkpoint = checkpoint[
            checkpoint["prompt_version"]
            == PROMPT_VERSION
        ].copy()

        checkpoint = checkpoint[
            checkpoint["entity_key"].isin(
                sample_eval["entity_key"]
            )
        ].drop_duplicates(
            "entity_key",
            keep="last",
        )

        print(
            "Checkpoint versi",
            PROMPT_VERSION,
            "ditemukan:",
            len(checkpoint),
        )
else:
    checkpoint = pd.DataFrame(
        columns=checkpoint_columns
    )
    print("Belum ada checkpoint V3.")

success_keys = set(
    checkpoint.loc[
        (
            checkpoint["status_api"]
            == "SUCCESS"
        )
        & (
            checkpoint["validation_status"]
            == "VALID"
        ),
        "entity_key",
    ]
)

pending = sample_eval[
    ~sample_eval["entity_key"].isin(
        success_keys
    )
].copy()

print("Sudah valid:", len(success_keys))
print("Belum diproses/gagal validasi:", len(pending))

Belum ada checkpoint V3.
Sudah valid: 0
Belum diproses/gagal validasi: 15


In [ ]:
# ============================================================
# 16. GENERATE REKOMENDASI
# ============================================================
for sequence, (_, row) in enumerate(
    pending.iterrows(),
    start=1,
):
    print(
        f"[{sequence}/{len(pending)}] "
        f"{row['title']} | {row['cluster_name']}"
    )

    prompt = build_prompt(row)

    result = generate_validated_recommendation(
        row
    )

    record = {
        "prompt_version": PROMPT_VERSION,
        "entity_key": row["entity_key"],
        "cluster_id": int(row["cluster_id"]),
        "cluster_name": row["cluster_name"],
        "title": row["title"],
        "categoryName": row["categoryName"],
        "totalScore": row["totalScore"],
        "reviewsCount": int(row["reviewsCount"]),
        "sentiment_score": int(
            row["sentiment_score"]
        ),
        "sentimen_confidence": (
            row["sentimen_confidence"]
        ),
        "nlp_review_flag": bool(
            row["nlp_review_flag"]
        ),
        "nlp_review_reason": (
            row["nlp_review_reason"]
            if pd.notna(
                row["nlp_review_reason"]
            )
            else ""
        ),
        "category_context": row[
            "category_context"
        ],
        "prompt_llm": prompt,
        **result,
    }

    checkpoint = checkpoint[
        checkpoint["entity_key"]
        != row["entity_key"]
    ]

    checkpoint = pd.concat(
        [
            checkpoint,
            pd.DataFrame([record]),
        ],
        ignore_index=True,
    )

    checkpoint[
        checkpoint_columns
    ].to_csv(
        OUTPUT_CHECKPOINT,
        index=False,
        sep=";",
        encoding="utf-8-sig",
        decimal=",",
    )

    print(
        "Status API:",
        result["status_api"],
        "| Validasi:",
        result.get(
            "validation_status",
            "",
        ),
    )

    time.sleep(1)

checkpoint = checkpoint[
    checkpoint_columns
].sort_values(
    ["cluster_id", "title"]
).reset_index(drop=True)

print("\nRingkasan status API:")
display(
    checkpoint["status_api"]
    .value_counts(dropna=False)
    .rename("jumlah")
    .to_frame()
)

print("\nRingkasan validasi otomatis:")
display(
    checkpoint["validation_status"]
    .value_counts(dropna=False)
    .rename("jumlah")
    .to_frame()
)

[1/15] XT komputer | Reputasi Positif–Volume Ulasan Rendah
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Status API: SUCCESS | Validasi: VALID


/tmp/ipykernel_4672/2798354416.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  checkpoint = pd.concat(


[2/15] BMW Motorrad DC Motorindo Bandung | Reputasi Positif–Volume Ulasan Rendah
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Status API: SUCCESS | Validasi: VALID
[3/15] Toserba Dm.Giftidea | Reputasi Positif–Volume Ulasan Rendah
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: length
Status API: SUCCESS | Validasi: VALID
[4/15] De & Dy | Reputasi Positif–Volume Ulasan Rendah
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: length
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Memanggil API, percobaan 1/4...
Respons berhasil diterima. Finish reason: stop
Status API: SUCCESS | Validasi: VALID
[5/15] Mutiara Salon | Reputasi Positif–Volume 

,jumlah
status_api,
SUCCESS,13
FAILED_VALIDATION,2



Ringkasan validasi otomatis:


,jumlah
validation_status,
VALID,13
INVALID,2


In [ ]:
# ============================================================
# 17. VALIDASI STRUKTUR JAWABAN
# ============================================================
required_sections = REQUIRED_SECTIONS

def validate_sections(text):
    text = str(text)

    missing_sections = [
        section
        for section in required_sections
        if section not in text
    ]

    return pd.Series({
        "struktur_lengkap": (
            len(missing_sections) == 0
        ),
        "bagian_hilang": " | ".join(
            missing_sections
        ),
    })


structure_check = checkpoint[
    "rekomendasi_llm"
].apply(
    validate_sections
)

checkpoint = pd.concat(
    [
        checkpoint.reset_index(drop=True),
        structure_check.reset_index(drop=True),
    ],
    axis=1,
)

failed_or_incomplete = checkpoint[
    (checkpoint["status_api"] != "SUCCESS")
    | (
        checkpoint["validation_status"]
        != "VALID"
    )
    | (
        ~checkpoint["struktur_lengkap"]
    )
].copy()

print("Struktur jawaban:")
display(
    checkpoint["struktur_lengkap"]
    .value_counts(dropna=False)
    .rename("jumlah")
    .to_frame()
)

if len(failed_or_incomplete) > 0:
    print(
        "Ada respons gagal, tidak valid, atau strukturnya tidak lengkap."
    )

    display(
        failed_or_incomplete[
            [
                "title",
                "status_api",
                "validation_status",
                "validation_errors",
                "error_api",
                "bagian_hilang",
            ]
        ]
    )
else:
    print(
        "Semua rekomendasi lolos API, validasi fakta, dan struktur."
    )

Struktur jawaban:


,jumlah
struktur_lengkap,
True,13
False,2


Ada respons gagal, tidak valid, atau strukturnya tidak lengkap.


,title,status_api,validation_status,validation_errors,error_api,bagian_hilang
2,Mutiara Salon,FAILED_VALIDATION,INVALID,Bagian tidak tersedia: [Analisis Konteks] | Bagian tidak tersedia: [Rekomendasi Strategi] | Bagian tidak tersedia: [Landasan Teori] | Bagian tidak tersedia: [Catatan Validitas ...,Respons gagal memenuhi validasi otomatis setelah tiga percobaan validasi.,[Analisis Konteks] | [Rekomendasi Strategi] | [Landasan Teori] | [Catatan Validitas Data]
6,Bengkel Motor & Aksesoris — Berkah Pit Station,FAILED_VALIDATION,INVALID,Bagian tidak tersedia: [Catatan Validitas Data],Respons gagal memenuhi validasi otomatis setelah tiga percobaan validasi.,[Analisis Konteks] | [Rekomendasi Strategi] | [Landasan Teori] | [Catatan Validitas Data]


In [ ]:
# ============================================================
# 18. RUBRIK DAN FORM EVALUASI MANUAL
# ============================================================
evaluation_columns = [
    "prompt_version",
    "entity_key",
    "cluster_id",
    "cluster_name",
    "title",
    "categoryName",
    "totalScore",
    "reviewsCount",
    "sentiment_score",
    "sentimen_confidence",
    "nlp_review_flag",
    "nlp_review_reason",
    "category_context",
    "status_api",
    "validation_status",
    "validation_errors",
    "response_model",
    "struktur_lengkap",
    "rekomendasi_llm",
]

df_evaluation = checkpoint[
    evaluation_columns
].copy()

for column in [
    "Skor_Kesesuaian_Cluster",
    "Skor_Spesifik_Kategori",
    "Skor_Relevansi_Teori",
    "Skor_Konsistensi_Data",
    "Skor_Kehati_Hatian_Data",
    "Skor_Kelayakan_Strategi",
    "Catatan_Evaluasi",
]:
    df_evaluation[column] = ""

rubric = pd.DataFrame({
    "Aspek": [
        "Kesesuaian dengan cluster",
        "Spesifik terhadap kategori",
        "Relevansi teori",
        "Konsistensi dengan data",
        "Kehati-hatian data",
        "Kelayakan strategi",
    ],
    "Skor 1": [
        "Tidak sesuai karakteristik cluster atau mengubah makna cluster.",
        "Sangat umum dan tidak mencerminkan kebutuhan kategori.",
        "Teori tidak relevan atau hubungannya tidak jelas.",
        "Terdapat angka, label, perbandingan, atau interpretasi yang bertentangan dengan input.",
        "Mengarang fakta, membuat klaim mutlak, atau mengabaikan kecukupan teks.",
        "Sulit dilakukan, tidak etis, atau tidak mempunyai indikator.",
    ],
    "Skor 2": [
        "Sebagian sesuai, tetapi masih ada arah yang kurang tepat.",
        "Cukup relevan, tetapi masih berlaku untuk banyak kategori.",
        "Teori cukup relevan, tetapi hubungan dengan tindakan belum kuat.",
        "Mayoritas sesuai, tetapi terdapat pernyataan ambigu atau kurang presisi.",
        "Cukup hati-hati, tetapi batasan data belum jelas.",
        "Cukup realistis, tetapi tindakan atau indikator perlu diperjelas.",
    ],
    "Skor 3": [
        "Sangat sesuai dan tidak mengubah hasil analisis.",
        "Menyebut kebutuhan operasional yang spesifik.",
        "Teori relevan dan mendukung tindakan dengan jelas.",
        "Seluruh angka, label, perbandingan, dan interpretasi konsisten dengan input.",
        "Tidak mengarang data dan mempertimbangkan kecukupan teks.",
        "Dua tindakan realistis, etis, dan mempunyai indikator.",
    ],
})

filling_guide = pd.DataFrame({
    "Kolom": [
        "Skor_Kesesuaian_Cluster",
        "Skor_Spesifik_Kategori",
        "Skor_Relevansi_Teori",
        "Skor_Konsistensi_Data",
        "Skor_Kehati_Hatian_Data",
        "Skor_Kelayakan_Strategi",
        "Catatan_Evaluasi",
    ],
    "Cara mengisi": [
        "Isi 1-3 berdasarkan kesesuaian dengan cluster semantik.",
        "Isi 1-3 berdasarkan kekhususan terhadap kategori usaha.",
        "Isi 1-3 berdasarkan ketepatan teori dan hubungannya.",
        "Isi 1-3 berdasarkan kesesuaian angka, label sentimen, perbandingan, dan interpretasi dengan input.",
        "Isi 1-3 berdasarkan kehati-hatian dan ketiadaan klaim dibuat-buat.",
        "Isi 1-3 berdasarkan realisme, etika, tindakan, dan indikator.",
        "Tulis alasan singkat, terutama untuk skor 1 atau 2.",
    ],
})

display(rubric)
display(filling_guide)

,Aspek,Skor 1,Skor 2,Skor 3
0,Kesesuaian dengan cluster,Tidak sesuai karakteristik cluster atau mengubah makna cluster.,"Sebagian sesuai, tetapi masih ada arah yang kurang tepat.",Sangat sesuai dan tidak mengubah hasil analisis.
1,Spesifik terhadap kategori,Sangat umum dan tidak mencerminkan kebutuhan kategori.,"Cukup relevan, tetapi masih berlaku untuk banyak kategori.",Menyebut kebutuhan operasional yang spesifik.
2,Relevansi teori,Teori tidak relevan atau hubungannya tidak jelas.,"Teori cukup relevan, tetapi hubungan dengan tindakan belum kuat.",Teori relevan dan mendukung tindakan dengan jelas.
3,Konsistensi dengan data,"Terdapat angka, label, perbandingan, atau interpretasi yang bertentangan dengan input.","Mayoritas sesuai, tetapi terdapat pernyataan ambigu atau kurang presisi.","Seluruh angka, label, perbandingan, dan interpretasi konsisten dengan input."
4,Kehati-hatian data,"Mengarang fakta, membuat klaim mutlak, atau mengabaikan kecukupan teks.","Cukup hati-hati, tetapi batasan data belum jelas.",Tidak mengarang data dan mempertimbangkan kecukupan teks.
5,Kelayakan strategi,"Sulit dilakukan, tidak etis, atau tidak mempunyai indikator.","Cukup realistis, tetapi tindakan atau indikator perlu diperjelas.","Dua tindakan realistis, etis, dan mempunyai indikator."


,Kolom,Cara mengisi
0,Skor_Kesesuaian_Cluster,Isi 1-3 berdasarkan kesesuaian dengan cluster semantik.
1,Skor_Spesifik_Kategori,Isi 1-3 berdasarkan kekhususan terhadap kategori usaha.
2,Skor_Relevansi_Teori,Isi 1-3 berdasarkan ketepatan teori dan hubungannya.
3,Skor_Konsistensi_Data,"Isi 1-3 berdasarkan kesesuaian angka, label sentimen, perbandingan, dan interpretasi dengan input."
4,Skor_Kehati_Hatian_Data,Isi 1-3 berdasarkan kehati-hatian dan ketiadaan klaim dibuat-buat.
5,Skor_Kelayakan_Strategi,"Isi 1-3 berdasarkan realisme, etika, tindakan, dan indikator."
6,Catatan_Evaluasi,"Tulis alasan singkat, terutama untuk skor 1 atau 2."


In [ ]:
# ============================================================
# 19. RINGKASAN DAN LOG API
# ============================================================
api_log = checkpoint[
    [
        "prompt_version",
        "entity_key",
        "title",
        "cluster_id",
        "status_api",
        "validation_status",
        "validation_errors",
        "validation_attempt",
        "attempt_count",
        "response_model",
        "generation_id",
        "error_api",
        "struktur_lengkap",
        "bagian_hilang",
    ]
].copy()

summary = pd.DataFrame([
    {
        "indikator": "Jumlah data clustering",
        "nilai": len(df),
    },
    {
        "indikator": "Jumlah cluster",
        "nilai": df["cluster_id"].nunique(),
    },
    {
        "indikator": "Sampel per cluster",
        "nilai": N_SAMPLE_PER_CLUSTER,
    },
    {
        "indikator": "Total rekomendasi",
        "nilai": len(checkpoint),
    },
    {
        "indikator": "Request API berhasil",
        "nilai": (
            checkpoint["status_api"]
            == "SUCCESS"
        ).sum(),
    },
    {
        "indikator": "Lolos validasi otomatis",
        "nilai": (
            checkpoint["validation_status"]
            == "VALID"
        ).sum(),
    },
    {
        "indikator": "Struktur lengkap",
        "nilai": (
            checkpoint["struktur_lengkap"]
            == True
        ).sum(),
    },
    {
        "indikator": "Model LLM",
        "nilai": MODEL_NAME,
    },
    {
        "indikator": "Prompt version",
        "nilai": PROMPT_VERSION,
    },
])

display(summary)

,indikator,nilai
0,Jumlah data clustering,3438
1,Jumlah cluster,3
2,Sampel per cluster,5
3,Total rekomendasi,15
4,Request API berhasil,13
5,Lolos validasi otomatis,13
6,Struktur lengkap,13
7,Model LLM,qwen/qwen-2.5-7b-instruct
8,Prompt version,V3_FACT_LOCKED


In [ ]:
# ============================================================
# 20. EKSPOR
# ============================================================
df_evaluation.to_csv(
    OUTPUT_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
)

df_evaluation.to_csv(
    OUTPUT_EVAL_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
)

rubric.to_csv(
    "05_Rubrik_Evaluasi.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
)

cluster_profile.to_csv(
    "05_Profil_Cluster.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
)

summary.to_csv(
    "05_Ringkasan.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
)

print("Berhasil membuat:")
print("-", OUTPUT_CSV)
print("-", OUTPUT_EVAL_CSV)
print("-", OUTPUT_CHECKPOINT)


Berhasil membuat:
- 05_Rekomendasi_LLM_15_Sampel_V3.csv
- 05_Hasil_Rekomendasi_dan_Evaluasi_LLM_V3.xlsx
- 05_Checkpoint_Rekomendasi_LLM_V3.csv


In [ ]:
# ============================================================
# 21. VALIDASI AKHIR
# ============================================================
assert len(df_evaluation) == EXPECTED_RECOMMENDATIONS
assert (
    df_evaluation["cluster_id"]
    .value_counts()
    .eq(5)
    .all()
)
assert df_evaluation["entity_key"].is_unique
assert (
    df_evaluation["prompt_version"]
    == PROMPT_VERSION
).all()

success_count = (
    df_evaluation["status_api"]
    == "SUCCESS"
).sum()

valid_count = (
    df_evaluation["validation_status"]
    == "VALID"
).sum()

complete_count = (
    df_evaluation["struktur_lengkap"]
    == True
).sum()

print("Jumlah rekomendasi:", len(df_evaluation))
print("Request API berhasil:", success_count)
print("Lolos validasi otomatis:", valid_count)
print("Struktur lengkap:", complete_count)
print("Prompt version:", PROMPT_VERSION)

if (
    success_count == EXPECTED_RECOMMENDATIONS
    and valid_count == EXPECTED_RECOMMENDATIONS
    and complete_count == EXPECTED_RECOMMENDATIONS
):
    print(
        "SEMUA REKOMENDASI BERHASIL, VALID, "
        "DAN SIAP DIEVALUASI MANUAL."
    )
else:
    print(
        "Masih ada request gagal, validasi gagal, "
        "atau struktur tidak lengkap. "
        "Jalankan ulang cell checkpoint dan batch."
    )

Jumlah rekomendasi: 15
Request API berhasil: 13
Lolos validasi otomatis: 13
Struktur lengkap: 13
Prompt version: V3_FACT_LOCKED
Masih ada request gagal, validasi gagal, atau struktur tidak lengkap. Jalankan ulang cell checkpoint dan batch.


In [ ]:
# ============================================================
# 22. DOWNLOAD
# ============================================================
if RUNNING_IN_COLAB:
    files.download(OUTPUT_CSV)
    files.download(OUTPUT_EVAL_CSV)
    files.download(OUTPUT_CHECKPOINT)
else:
    print("File tersimpan pada folder kerja.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Narasi metodologi

> Large Language Model digunakan setelah proses clustering selesai untuk membantu memformulasikan rekomendasi berdasarkan profil cluster, kategori usaha, rating, volume ulasan Google Maps, sentiment score, dan tingkat kecukupan teks. LLM tidak digunakan untuk menentukan cluster atau mengubah hasil analisis. Sebanyak lima listing dari setiap cluster dipilih sebagai sampel evaluasi. Rekomendasi dinilai secara manual menggunakan rubrik kesesuaian cluster, spesifisitas kategori, relevansi teori, kehati-hatian terhadap keterbatasan data, dan kelayakan strategi.

## Batasan

- Rekomendasi LLM merupakan hasil formulasi, bukan keputusan bisnis mutlak.
- Evaluasi manusia tetap diperlukan.
- Volume ulasan Google Maps bukan ukuran jumlah pelanggan, kunjungan, transaksi, penjualan, omzet, atau tingkat keramaian offline.
- Cluster `Reputasi Positif–Volume Ulasan Rendah` tidak boleh ditafsirkan sebagai kelompok usaha yang sepi atau kurang populer secara offline.
- Cluster `Performa Digital Tinggi` menggambarkan performa pada data Google Maps, bukan dominasi pasar.
- Rekomendasi tidak menggambarkan kondisi internal, omzet, laba, atau pangsa pasar.
- Hasil sentimen tetap memiliki keterbatasan pada bahasa daerah, slang, jumlah teks, dan konteks lokal.